In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect('database.sqlite')

In [2]:
CATEGORICAL_COLUMNS = [
    'attackComplexity',
    'attackVector',
    'availabilityImpact',
    'baseSeverity',
    'confidentialityImpact',
    'integrityImpact',
    'privilegesRequired',
    'scope',
    'userInteraction'
]

In [3]:
dfs = []
for year in [2022, 2023, 2024, 2025]:
    try:
        df_year = pd.read_sql_query(f"SELECT * FROM YEAR_{year}", con)
        df_year['source_year'] = year  # Keep track of the year group
        dfs.append(df_year)
    except Exception as e:
        print(f"Warning: Could not read table YEAR_{year}: {e}")

db = pd.concat(dfs, ignore_index=True)
db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121421 entries, 0 to 121420
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   cveId                  121421 non-null  object 
 1   dataVersion            121421 non-null  float64
 2   datePublished          121421 non-null  object 
 3   product                105451 non-null  object 
 4   vendor                 101683 non-null  object 
 5   cweId                  111256 non-null  object 
 6   attackComplexity       121421 non-null  object 
 7   attackVector           121421 non-null  object 
 8   availabilityImpact     121421 non-null  object 
 9   baseScore              121421 non-null  float64
 10  baseSeverity           121421 non-null  object 
 11  confidentialityImpact  121421 non-null  object 
 12  integrityImpact        121421 non-null  object 
 13  privilegesRequired     121421 non-null  object 
 14  scope                  121421 non-nu

In [4]:
def display_yearly_breakdown(df, column_name):
    # crosstab puts the feature on rows and source_year on columns
    breakdown = pd.crosstab(df[column_name], df['source_year']).fillna(0).astype(int)

    # Replace top left corner with the column name for clarity
    breakdown.index.name = None
    breakdown.columns.name = column_name
            
    # Sort columns chronologically
    breakdown = breakdown[sorted(breakdown.columns)]
    display(breakdown)

In [5]:
for col in CATEGORICAL_COLUMNS:
    if col in db.columns:
        display_yearly_breakdown(db, col)

attackComplexity,2022,2023,2024,2025
High,2189,2664,3455,4292
Low,15303,22121,33609,37788


attackVector,2022,2023,2024,2025
Adjacent,635,1156,1429,1248
Local,5380,5956,7058,7270
Network,11279,17444,28208,33128
Physical,198,229,369,434


availabilityImpact,2022,2023,2024,2025
High,9396,11149,14841,15116
Low,1977,4563,7190,11046
None,6119,9073,15033,15918


baseSeverity,2022,2023,2024,2025
Critical,1849,2473,3788,3818
High,6800,9001,13056,14743
Low,876,1296,1390,1617
Medium,7965,12011,18818,21861
None,2,4,12,41


confidentialityImpact,2022,2023,2024,2025
High,9069,11696,16239,16957
Low,3968,6983,13038,15568
None,4455,6106,7787,9555


integrityImpact,2022,2023,2024,2025
High,7865,10044,13559,14014
Low,4137,7998,14218,17118
None,5490,6743,9287,10948


privilegesRequired,2022,2023,2024,2025
High,2504,3557,4032,4048
Low,6275,8969,14761,16189
None,8713,12259,18271,21843


scope,2022,2023,2024,2025
Changed,3304,5565,10344,11236
Unchanged,14188,19220,26720,30844


userInteraction,2022,2023,2024,2025
None,11879,16337,25494,29276
Required,5613,8448,11570,12804


In [6]:
# Only numeric columns are baseScore
score_breakdown = pd.crosstab(db['baseScore'], db['source_year']).fillna(0).astype(int)

# Replace top left corner with the column name for clarity
score_breakdown.index.name = None
score_breakdown.columns.name = 'baseScore'

display(score_breakdown[sorted(score_breakdown.columns)])

baseScore,2022,2023,2024,2025
0.0,2,4,12,41
0.1,0,0,2,0
0.3,0,0,0,1
0.5,0,0,0,2
0.6,0,0,0,12
...,...,...,...,...
9.6,85,96,121,80
9.7,1,21,13,22
9.8,1199,1615,2169,1806
9.9,84,95,144,166
